In [1]:
# Work around a TRL 0.24.0 + Transformers 5.5.0 bug where prompt-only apply_chat_template() returns a BatchEncoding, causing TRL to miscompute the prompt length (e.g., len(prompt_ids) == 2) and incorrectly mask only the first few prompt tokens.
# SFT caompatibe != TRL 0.24.0 & Transformers 5.5 -> prompt-only apply_chat_template() error reported
%pip install --upgrade --no-cache-dir transformers trl datasets peft accelerate bitsandbytes safetensors
!pip install --upgrade "torchao>0.16.0" # PEFT requirement

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 845.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 384.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 141.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 75.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
  from google.colab import drive

  drive.mount('/content/drive') # load google drive
  os.chdir('/content/drive/My Drive/Colab_Notebooks') # change directory to the current working directory

Mounted at /content/drive


In [ ]:
from huggingface_hub import notebook_login
from huggingface_hub import login
notebook_login()

login(token="[HF Token]")

In [4]:
import pandas as pd
import json

SFT_TRAIN_DATA_LOAD_PATH = "mathdial_unused_rows__student_teacher_pairs_df.csv"
TEST_DATA_LOAD_PATH = "train_test_split/test_stepverify_labeled_0.9.json" # Same as DPO

#model_id
BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

SFT_ADAPTER_PATH = "llama3-8b-instruct-sft-adapter" # Adapter name
SFT_LOCAL_ADAPTER_DIR = os.path.join("/content", SFT_ADAPTER_PATH) # save in the current colab loacl disk ( circumvent google drive i/o limit )
DRIVE_ROOT_DIR = "/content/drive/My Drive/Colab_Notebooks" # current notebook directory in the google drive
SFT_DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT_DIR, SFT_ADAPTER_PATH)    # save in the ADAPTER directory in the google drive

SEED = 42

#### Load data

In [5]:
data = pd.read_csv(SFT_TRAIN_DATA_LOAD_PATH).to_dict(orient="records")
test_data = json.load(open(TEST_DATA_LOAD_PATH, "r"))
test_data = pd.DataFrame(test_data).rename(columns={"student_incorrect_solution": "student_mistake"}).to_dict(orient="records") # change the key name to match pref_data_format

#### Train val test split

In [6]:
from sklearn.model_selection import train_test_split
import pandas as pd


def split_train_test(data, random_state=SEED):
    indices = list(range(len(data)))
    train_data, val_data, train_idx, val_idx = train_test_split(
        data,
        indices,
        test_size=0.20,
        random_state=SEED,
        shuffle=True,
    )
    return train_data, val_data, train_idx, val_idx


In [7]:
train_data, val_data, train_idx, val_idx = split_train_test(data)

#### Apply chat template and load dataset

In [8]:
from datasets import Dataset

SYSTEM_TAMPLATE = """You are an experienced elementary mathematics tutor. Your role is not merely to correct the student's mistake. Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
You will be given a problem from a elementary mathematics lesson. Your task is to guide the student's response to the problem.

Your response should be at most two sentences.
"""

USER_TEMPLATE = """
### Problem:
{problem}

### Student's response:
{student}
"""


def make_sft_train_val_dataset(data):
  dataset = Dataset.from_list([
      {
          "prompt": [
              {
                  "role": "system",
                  "content": SYSTEM_TAMPLATE,
              },
              {
                  "role": "user",
                  "content": USER_TEMPLATE.format(
                      problem = str(row["problem"]),
                      student = str(row["student"])
                  ),
              },
          ],
          "completion": [
              {
                  "role": "assistant",
                  "content": str(row["teacher"]),
              }
          ],
      }
      for row in data
  ])
  return dataset


def make_sft_test_dataset(data):
  dataset = Dataset.from_list([
      {
          "prompt": [
              {
                  "role": "system",
                  "content": SYSTEM_TAMPLATE,
              },
              {
                  "role": "user",
                  "content": USER_TEMPLATE.format(
                      problem = str(row["problem"]),
                      student = str(row['student_mistake'])
                  ),
              },
          ],
          "completion": [
              {
                  "role": "assistant",
                  # "content": "(" + str(row['dialog_history'][0]['pedagogy']) + ")" + str(row['dialog_history'][0]['text']),
                  "content": str(row['dialog_history'][0]['text']),
              }
          ],
      }
      for row in data
  ])
  return dataset


In [9]:
train_ds = make_sft_train_val_dataset(train_data)
val_ds = make_sft_train_val_dataset(val_data)
test_ds = make_sft_test_dataset(test_data)

print(f"Train: {len(train_ds)} ({len(train_data)/len(data):.1%})")
print(f"Valid: {len(val_ds)} ({len(val_data)/len(data):.1%})")
print(f"Test:  {len(test_ds)} ({len(test_data)/len(data):.1%})")


Train: 8111 (80.0%)
Valid: 2028 (20.0%)
Test:  298 (2.9%)


#### Load model and prepare LoRA

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer #, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer # , DataCollatorForCompletionOnlyLM -> replaced by completion_only_loss=True ( https://github.com/huggingface/trl/discussions/3826 )
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model


# LoRA Config
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,                          # rank
    lora_alpha=32,                 # alpha = 2 * r (recommended ratio in practice)
    # target_modules="all_layers",   # QLoRA setting
    target_modules=["q_proj", "k_proj", "o_proj", "v_proj"],   # apply only to Key, Query, Value, Output weights ( Q, V are the most important )
    lora_dropout=0.05,             # DPO Unsloth -> no drop_out is desirable
    bias="none",
  )


# Load Toeknizer
sft_tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID
    )
sft_tokenizer.pad_token = sft_tokenizer.eos_token
sft_tokenizer.padding_side = "left"


#  QLoRA - 4-bit Model Quantization ( in case the memory explodes )
# bnb_config = BitsAndBytesConfig(
#   load_in_4bit=True,                        # Use 4-bit precision model loading
#   bnb_4bit_quant_type="nf4",                # Quantization type
#   bnb_4bit_compute_dtype=torch.bfloat16,    # Compute dtype
#   bnb_4bit_use_double_quant=True,           # Apply double qunatization
# )


# Load base model
sft_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    # quantization_config=bnb_config # QLoRA setting
)
sft_model.config.use_cache = False
sft_model.config.pad_token_id = sft_tokenizer.pad_token_id

# model = get_peft_model(model, lora_config) # attach a new LoRA adapter to the model, where the model will be modified in place
# Three Ways to configure PERFT : https://huggingface.co/docs/trl/peft_integration?utm_source=chatgpt.com
# When passing LoraConfig to SFTTrainer, it is not necessary to call get_peft_model() directly, as the latest TRL officially supports passing peft_config to the Trainer

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

#### SFT Config

In [11]:
from trl import SFTConfig, SFTTrainer # , DataCollatorForCompletionOnlyLM -> replaced by completion_only_loss=True ( https://github.com/huggingface/trl/discussions/3826 )

sft_training_args = SFTConfig(
    output_dir=SFT_LOCAL_ADAPTER_DIR,  # write into the current colab disk

    # SFT, Very Important! Compute the loss of completion only ( prompts are ignored during training )
    completion_only_loss=True, # When set True, no need to manually mark -100 for prompt. SFTTrainer automatically apply the tokenizer's apply_chat_template to the message and tokenize it. ( Prompt -> -100 )


    # Batch
    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    #gradient_accumulation_steps=4,

    max_length=1024,


    # Optimization
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=0.05, # 5 % of the total steps
    bf16=True,

    train_sampling_strategy="group_by_length", #  group_by_lenght latest
    # optim="paged_adamw_32bit",        # Paged Optimizer - recommended optimizer for QLoRA, which utilizes CUDA unified memory to page optimizer state in case GPU reaches the limit


    # Evaluation / logging
    eval_strategy="steps",
    eval_steps=200,

    logging_steps=25,
    save_strategy="steps",
    save_steps=200,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",


    # Gradient chckpointing
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_training_args,

    train_dataset=train_ds,
    eval_dataset=val_ds,

    processing_class=sft_tokenizer,  # tokenizer uses its apply_chat_template to the input messages ( prompt tokens = -100)
    peft_config=lora_config  # LoRA Config : https://huggingface.co/docs/trl/peft_integration?utm_source=chatgpt.com
)

Tokenizing train dataset:   0%|          | 0/8111 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8111 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8111 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8111 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2028 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/2028 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2028 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/2028 [00:00<?, ? examples/s]

In [12]:
sft_trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.916352,1.917068,1.913995,744884.000000,0.534447
400,1.944643,1.857118,1.914176,1491742.000000,0.542949
600,1.784050,1.836759,1.727929,2242195.000000,0.546782
800,1.841668,1.818749,1.776121,2988794.000000,0.549249
1000,1.807758,1.809718,1.795218,3733421.000000,0.551488
1200,1.831874,1.807242,1.782749,4481547.000000,0.551596
1400,1.751789,1.805509,1.769994,5225654.000000,0.551929
1521,1.790483,1.805569,1.770697,5666778.000000,0.552332


TrainOutput(global_step=1521, training_loss=1.9206308268309424, metrics={'train_runtime': 1290.3574, 'train_samples_per_second': 18.858, 'train_steps_per_second': 1.179, 'total_flos': 2.5923211739529216e+17, 'train_loss': 1.9206308268309424, 'epoch': 3.0})

#### Save model

In [13]:

import os
import shutil

# 1. Save LoRA adapter
# trainer.save_model(LOCAL_ADAPTER_DIR)
sft_trainer.model.save_pretrained(SFT_LOCAL_ADAPTER_DIR)

# 2. Save Tokenizer
sft_tokenizer.save_pretrained(SFT_LOCAL_ADAPTER_DIR)


print("Saved to local drive:", SFT_LOCAL_ADAPTER_DIR)

# 3. Google Drive에 폴더 전체 복사
shutil.copytree(
    SFT_LOCAL_ADAPTER_DIR,
    SFT_DRIVE_MODEL_DIR,
    dirs_exist_ok=True,
)

print("Saved to Google Drive:", SFT_DRIVE_MODEL_DIR)

Saved to local drive: /content/llama3-8b-instruct-sft-adapter
Saved to Google Drive: /content/drive/My Drive/Colab_Notebooks/llama3-8b-instruct-sft-adapter


---

#### Inference

In [14]:
from transformers import pipeline

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto"
)

base_tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID
    )
base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "left"


sft_model = AutoModelForCausalLM.from_pretrained( # adapter_config.json 읽고 base model 다 가져옴
    SFT_DRIVE_MODEL_DIR,
    dtype=torch.bfloat16,
    device_map="auto")

sft_tokenizer = AutoTokenizer.from_pretrained(
    SFT_DRIVE_MODEL_DIR,
    clean_up_tokenization_spaces=False,
    )


# inference mode
base_model.eval()
base_model.config.use_cache = True

sft_model.eval()
sft_model.config.use_cache = True



# 그리고 SFT_DRIVE_MODEL_DIR의 LoRA adapter를 attach

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

In [15]:
# Fine-tuned Model Response
example_prompt = sft_tokenizer.apply_chat_template(
    test_ds[50]["prompt"],
    tokenize=False,
    add_generation_prompt=True) # 현재 Transformers의 text-generation pipeline은 messages 형태를 직접 받을 수 있고, chat template도 pipeline이 알아서 적용합니다 https://github.com/huggingface/transformers/blob/main/src/transformers/pipelines/text_generation.py?utm_source=chatgpt.com

base_model_pipe = pipeline(
    task="text-generation",
    model=base_model,
    tokenizer=base_tokenizer,
    return_full_text=False)

sft_pipe = pipeline(
    task="text-generation",
    model=sft_model,
    tokenizer=sft_tokenizer,
    return_full_text=False)

# apply_chat_template
# add_generation_prompt=True is needed for the model to generate the next tokens as trained : https://huggingface.co/docs/transformers/chat_templating

print("=" * 200)
print(" [ Prompt ] \n\n")
print(example_prompt)

print("=" * 200)

print( " [ Ground Truth ] ")
print(test_ds[100]['completion'][0]['content'])

print("=" * 200)

print(" [ Base_Response ]")
print(base_model_pipe(example_prompt)[0]["generated_text"])

print("=" * 200)

print("[ Fine-tuned_Response ]")
print(sft_pipe(example_prompt)[0]["generated_text"])

 [ Prompt ] 


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an experienced elementary mathematics tutor. Your role is not merely to correct the student's mistake. Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
You will be given a problem from a elementary mathematics lesson. Your task is to guide the student's response to the problem.

Your response should be at most two sentences.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Problem:
pirate rick sailed his ship to a tropical island in search of a site to bury his treasure. after finding the perfect site, it took him 4 hours to dig up 8 feet of sand under which to bury the treasure. once the treasure was buried, he left the island. then, a tropical storm came and washed away half of the sand from on top of the treasure. next, a giant tsunami wave poured over the island, adding 2 feet of new sand back onto the site of his treasure. when pirate

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I love the effort! You're on the right track! However, let's break down the problem step by step. After the tsunami, there were indeed 4 + 2 = 6 feet of sand on top of the treasure, not 4 2 6 feet. And then, to find the total amount of sand Pirate Rick needs to dig through, we need to add the 6 feet to the original 8 feet, making it 14 feet.
[ Fine-tuned_Response ]
can you tell me how much sand is there on top of the treasure after the tsunami?


In [16]:

test_ds[0]['completion'][0]['content']

'hi brenda, could you please walk me through your solution?'

---

#### Chat Template Check ( check batch tokens )

In [17]:
batch = next(iter(sft_trainer.get_train_dataloader()))

input_ids = batch["input_ids"][0]
labels = batch["labels"][0] # tokenized

loss_mask = labels != -100
label_mask = labels == -100

print("Total tokens:", input_ids.numel())
print("Toekns for loss:", loss_mask.sum().item())

target_token_ids = input_ids[loss_mask].tolist()
label_token_ids = input_ids[label_mask].tolist()


print("=" * 200)


# Prompt, Loss is not calculated
print("\nlabel:")
print(sft_trainer.processing_class.decode(
label_token_ids,
        skip_special_tokens=False,
    )
)
print("label tokens:")
print(labels[label_mask])


print("=" * 200)

# Assistant Token Only, Loss is calculated
print("\nLoss target:")
print(
    sft_trainer.processing_class.decode(
        target_token_ids,
        skip_special_tokens=False,
    )
)

print("\nLoss target tokens:")
print(labels[loss_mask])


Total tokens: 481
Toekns for loss: 49

label:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an experienced elementary mathematics tutor. Your role is not merely to correct the student's mistake. Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
You will be given a problem from a elementary mathematics lesson. Your task is to guide the student's response to the problem.

Your response should be at most two sentences.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Problem:
Lou Senior took 3 cookies out of the cookie jar and ate them.  Since he didn't get caught by his wife, he went back the next day and took another 3 cookies out of the jar.  But after eating just one of the cookies, he felt guilty about it and put the other two cookies back.  His son, Louie Junior saw that his Dad was eating cookies.  So, Louie Junior took seven cookies out of the jar and hid them in his bedroom for later.  The next

#### LoRA check

In [18]:
from peft import PeftModel, get_model_status

model = sft_trainer.model

print("Base Model:", model.peft_config["default"].base_model_name_or_path)

print("\nModel type:", type(model))
print("Is PeftModel:", isinstance(model, PeftModel))

print("\n=== PEFT status ===")
status = get_model_status(model)
print(status)

print("\n=== Trainable parameters ===")
model.print_trainable_parameters()

trainable_params = [name for name, param in sft_trainer.model.named_parameters() if param.requires_grad]
print("\nNumber of trainable tensors (first 20 tensors):", len(trainable_params))
for name in trainable_params[:20]:
    print(name)

Base Model: meta-llama/Meta-Llama-3-8B-Instruct

Model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Is PeftModel: True

=== PEFT status ===
TunerModelStatus(base_model_type='LlamaForCausalLM', adapter_model_type='LoraModel', peft_types={'default': 'LORA'}, trainable_params=13631488, total_params=8043892736, num_adapter_layers=128, enabled=True, active_adapters=['default'], merged_adapters=[], requires_grad={'default': True}, available_adapters=['default'], devices={'default': ['cuda']}, quantization_backend=None)

=== Trainable parameters ===
trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695

Number of trainable tensors (first 20 tensors): 256
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.model.lay